# VNINDEX Yang-Zhang Volatility Regime

Plot VNINDEX and classify volatility regime using rolling percentiles of Yang-Zhang variance.

In [10]:
import warnings
warnings.filterwarnings('ignore')

import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os, sys, pathlib

os.environ.setdefault('NUMBA_CACHE_DIR', str(pathlib.Path('.numba_cache').resolve()))
os.environ.setdefault('MPLCONFIGDIR', str(pathlib.Path('.mplconfig').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(pathlib.Path('.cache').resolve()))
pathlib.Path(os.environ['NUMBA_CACHE_DIR']).mkdir(parents=True, exist_ok=True)
pathlib.Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
pathlib.Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(pathlib.Path('..').resolve()))
from backend.app.services.indicators.gkyz_volatility import calculate_gkyz_volatility
from backend.app.services.data_loader import load_stocks


ANNUALISE = 252
SYMBOL = 'VNINDEX'
WINDOW = 20
LOOKBACK = 252

plt.style.use('dark_background')
print('imports ok')

imports ok


In [ ]:
df = load_stocks(refresh=False)

open_  = df['open']
high   = df['high']
low    = df['low']
close  = df['close']
volume = df['volume']

df_all = df

vn = pd.DataFrame({
    'open': df['open'][SYMBOL],
    'high': df['high'][SYMBOL],
    'low': df['low'][SYMBOL],
    'close': df['close'][SYMBOL],
}).sort_index()
vn = vn.dropna()

HDF5 not found — fetching from DeltaLake …
Loaded 985 symbols × 2499 bars from DeltaLake
Saved to stocks_data_latest.h5
Shape: (2499, 985)  |  2016-08-17 → 2026-08-14


In [12]:
def yz_daily_components(open_: pd.Series, high: pd.Series, low: pd.Series, close: pd.Series, window: int):
    """Single-period Yang-Zhang variance components."""
    log_co = np.log(close / open_)
    log_oc = np.log(open_ / close.shift(1))
    rs = (
        np.log(high / close) * np.log(high / open_)
        + np.log(low / close) * np.log(low / open_)
    )
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    return (log_oc ** 2), (log_co ** 2), rs, k

def ht_factor(window: int) -> float:
    """Hodges-Tompkins correction factor for rolling-volatility bias."""
    if window <= 1:
        return 1.0
    return 1.0 / (1.0 - (window / ANNUALISE) + ((window**2 - 1) / (3.0 * ANNUALISE**2)))

sig_o2, sig_c2, sig_rs2, k = yz_daily_components(vn['open'], vn['high'], vn['low'], vn['close'], WINDOW)
vn['yz_var_daily'] = (sig_o2 + k * sig_c2 + (1 - k) * sig_rs2).clip(lower=0)

# Step 2: roll and annualize from daily YZ variance
vn['yz_var_roll_sum'] = vn['yz_var_daily'].rolling(WINDOW).sum()
vn['yz_vol'] = np.sqrt((ANNUALISE / WINDOW) * vn['yz_var_roll_sum']) * ht_factor(WINDOW)

# Step 3-4: past-only reference distribution and percentile-rank regime
hist = vn['yz_vol'].shift(1)
p25 = hist.rolling(LOOKBACK).quantile(0.25)
p75 = hist.rolling(LOOKBACK).quantile(0.75)
p90 = hist.rolling(LOOKBACK).quantile(0.90)

def trailing_percentile_rank(series: pd.Series, lookback: int) -> pd.Series:
    vals = series.to_numpy(dtype=float)
    out = np.full(vals.shape, np.nan)
    for i in range(lookback, len(vals)):
        cur = vals[i]
        ref = vals[i - lookback:i]
        ref = ref[np.isfinite(ref)]
        if np.isfinite(cur) and ref.size > 0:
            out[i] = (ref < cur).mean() * 100.0
    return pd.Series(out, index=series.index)

vn['pct_rank'] = trailing_percentile_rank(vn['yz_vol'], LOOKBACK)

regime = pd.Series('Normal', index=vn.index)
regime[vn['pct_rank'] < 25] = 'Low'
regime[vn['pct_rank'] > 75] = 'High'
regime[vn['pct_rank'] > 90] = 'Crisis'
vn['regime'] = regime

vn[['yz_var_daily', 'yz_var_roll_sum', 'yz_vol', 'pct_rank', 'regime']].tail()

,yz_var_daily,yz_var_roll_sum,yz_vol,pct_rank,regime
date,,,,,
2026-08-10,0.000123,0.002967,0.209542,43.650794,Normal
2026-08-11,0.000072,0.002849,0.205316,35.714286,Normal
2026-08-12,0.000056,0.002821,0.204330,34.523810,Normal
2026-08-13,0.000079,0.002632,0.197362,28.174603,Normal
2026-08-14,0.000207,0.002823,0.204385,35.317460,Normal


In [13]:
plot_df = vn.dropna(subset=['yz_vol', 'pct_rank']).copy()
p25_plot = p25.reindex(plot_df.index)
p75_plot = p75.reindex(plot_df.index)
p90_plot = p90.reindex(plot_df.index)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.62, 0.38],
    subplot_titles=(
        'VNINDEX with Yang-Zhang Volatility Regime',
        f'Yang-Zhang Volatility ({WINDOW}d, HT corrected) with Past-{LOOKBACK}d Percentile Bands',
    ),
)

fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df['close'], name='VNINDEX', line=dict(color='white', width=1.1)),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df['yz_vol'] * 100, name='YZ Vol (ann %)', line=dict(color='#f7c59f', width=1.2)),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=p25_plot * 100, name='25th pct', line=dict(color='#66bb6a', width=1, dash='dash')),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=p75_plot * 100, name='75th pct', line=dict(color='#ef5350', width=1, dash='dash')),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=p90_plot * 100, name='90th pct', line=dict(color='#ff1744', width=1, dash='dot')),
    row=2, col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=p75_plot * 100,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    ),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=p25_plot * 100,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(100,181,246,0.12)',
        name='25-75 pct band',
        hoverinfo='skip',
    ),
    row=2, col=1,
)

color_map = {
    'Crisis': 'rgba(255,23,68,0.10)',
    'High': 'rgba(239,83,80,0.08)',
    'Normal': 'rgba(100,181,246,0.05)',
    'Low': 'rgba(102,187,106,0.08)',
}
idx = plot_df.index
reg = plot_df['regime']
start = idx[0]
cur = reg.iloc[0]
for i in range(1, len(idx)):
    if reg.iloc[i] != cur:
        fig.add_vrect(x0=start, x1=idx[i], fillcolor=color_map.get(cur, 'rgba(158,158,158,0.05)'), line_width=0, layer='below', row=1, col=1)
        start = idx[i]
        cur = reg.iloc[i]
fig.add_vrect(x0=start, x1=idx[-1], fillcolor=color_map.get(cur, 'rgba(158,158,158,0.05)'), line_width=0, layer='below', row=1, col=1)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    height=820,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.18, x=0, xanchor='left'),
    margin=dict(t=70, b=110, l=60, r=40),
)
fig.update_yaxes(title_text='Index', tickformat=',.0f', row=1, col=1)
fig.update_yaxes(title_text='Annualised Vol %', ticksuffix='%', row=2, col=1)
fig.update_xaxes(matches='x', row=1, col=1)
fig.update_xaxes(matches='x', rangeslider_visible=True, row=2, col=1)

fig.show()

In [14]:
regime_share = (plot_df['regime'].value_counts(normalize=True) * 100).round(1)
print('Regime distribution (% of available days):')
for k in ['Crisis', 'High', 'Normal', 'Low']:
    print(f'  {k:6s}: {regime_share.get(k, 0.0):5.1f}%')

latest = plot_df.iloc[-1]
print('\nLatest snapshot:')
print(f"  Date         : {plot_df.index[-1].date()}")
print(f"  Close        : {latest['close']:.2f}")
print(f"  YZ Variance (daily) : {latest['yz_var_daily']:.8f}")
print(f"  Percentile Rank     : {latest['pct_rank']:.1f}")
print(f"  YZ Vol (ann) : {latest['yz_vol']*100:.2f}%")
print(f"  Regime       : {latest['regime']}")

Regime distribution (% of available days):
  Crisis:  12.9%
  High  :  13.8%
  Normal:  46.0%
  Low   :  27.3%

Latest snapshot:
  Date         : 2026-08-14
  Close        : 1729.08
  YZ Variance (daily) : 0.00020657
  Percentile Rank     : 35.3
  YZ Vol (ann) : 20.44%
  Regime       : Normal


In [17]:
# ── Defaults — can be overridden in Optuna ────────────────────────────────────
GKYZ_WINDOW = 21
GKYZ_UPPER  = 0.8   # threshold to enter risk-on  (high-vol)
GKYZ_LOWER  = 0.2   # threshold to exit  risk-on  (low-vol)

def _value_on_or_before(s: pd.Series, day, *, dropna: bool = False):
    ts = pd.to_datetime(day, dayfirst=True).normalize()
    sub = s.loc[:ts]
    if dropna:
        sub = sub.dropna()
    if sub.empty:
        raise ValueError(f'No data on or before {ts.date()}')
    idx = sub.index[-1]
    exact = pd.Timestamp(idx).normalize() == ts
    return idx, sub.iloc[-1], exact

def compute_gkyz_regime(
    open_s: pd.Series,
    high_s: pd.Series,
    low_s:  pd.Series,
    close_s: pd.Series,
    window: int   = GKYZ_WINDOW,
    upper:  float = GKYZ_UPPER,
    lower:  float = GKYZ_LOWER,
) -> tuple:
    """
    Returns (gkyz_series, risk_on_series) both aligned to close_s.index.

    risk_on = True  → high-vol regime; risk_on = False → low-vol regime.
    Map to fractional size with GKYZ_SIZE_TREND (v1/v2) or GKYZ_SIZE_MR (v3).
    """
    gkyz_arr = calculate_gkyz_volatility(
        open_s.values.astype(np.float64),
        high_s.values.astype(np.float64),
        low_s.values.astype(np.float64),
        close_s.values.astype(np.float64),
        window=window,
        normalize=True,
    )

    n = len(gkyz_arr)
    risk_on = np.zeros(n, dtype=bool)
    state   = False                         # start in risk-off
    for i in range(n):
        v = gkyz_arr[i]
        if np.isnan(v):
            risk_on[i] = state
            continue
        if not state and v > upper:         # cross above → risk-on
            state = True
        elif state and v < lower:           # cross below → risk-off
            state = False
        risk_on[i] = state

    idx = close_s.index
    return pd.Series(gkyz_arr, index=idx, name='gkyz'), \
           pd.Series(risk_on,  index=idx, name='risk_on')


# Compute once on VNINDEX
gkyz_vnindex, risk_on_vnindex = compute_gkyz_regime(
    vn['open'], vn['high'], vn['low'], vn['close'],
)

STOCK_SYMBOL = 'PVS'
GKYZ_LOOKUP_DAY = '14/08/2026'

_sym = pd.DataFrame({
    'open': df_all['open'][STOCK_SYMBOL],
    'high': df_all['high'][STOCK_SYMBOL],
    'low': df_all['low'][STOCK_SYMBOL],
    'close': df_all['close'][STOCK_SYMBOL],
}).dropna().sort_index()
gkyz_symbol, risk_on_symbol = compute_gkyz_regime(
    _sym['open'], _sym['high'], _sym['low'], _sym['close'],
)

n_risk_on  = risk_on_vnindex.sum()
n_risk_off = (~risk_on_vnindex).sum()
print(f'Risk-on  bars: {n_risk_on}  ({n_risk_on / len(risk_on_vnindex) * 100:.1f}%)')
print(f'Risk-off bars: {n_risk_off} ({n_risk_off / len(risk_on_vnindex) * 100:.1f}%)')

# GKYZ at a chosen day: last row on or before that calendar date (handles holidays)


vn_idx, vn_val, vn_exact = _value_on_or_before(gkyz_vnindex, GKYZ_LOOKUP_DAY, dropna=True)
sy_idx, sy_val, sy_exact = _value_on_or_before(gkyz_symbol, GKYZ_LOOKUP_DAY, dropna=True)
vn_ridx, vn_risk, _ = _value_on_or_before(risk_on_vnindex, GKYZ_LOOKUP_DAY)
sy_ridx, sy_risk, _ = _value_on_or_before(risk_on_symbol, GKYZ_LOOKUP_DAY)

print(f'\nGKYZ @ {GKYZ_LOOKUP_DAY} (normalized percentile in [0,1])')
print(f'  VNINDEX  row={vn_idx.date()}  gkyz={float(vn_val):.6f}  risk_on={bool(vn_risk)}  exact_calendar_day={vn_exact}')
print(f'  {STOCK_SYMBOL:7s} row={sy_idx.date()}  gkyz={float(sy_val):.6f}  risk_on={bool(sy_risk)}  exact_calendar_day={sy_exact}')

Risk-on  bars: 1152  (46.1%)
Risk-off bars: 1347 (53.9%)

GKYZ @ 14/08/2026 (normalized percentile in [0,1])
  VNINDEX  row=2026-08-14  gkyz=0.733082  risk_on=True  exact_calendar_day=True
  PVS     row=2026-08-14  gkyz=0.389227  risk_on=True  exact_calendar_day=True
